<a href="https://colab.research.google.com/github/psychic-coder/WASHING-MACHINE-FUZZY-LOGIC-SYSTEM/blob/main/WASHING_MACHINE_FUZZY_LOGIC_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# WASHING MACHINE FUZZY LOGIC SYSTEM
# For Google Colab
# ============================================

# Step 1: Install required libraries (run this once)
!pip install scikit-fuzzy matplotlib numpy

# Step 2: Import libraries
import numpy as np
import skfuzzy as fuzz
import matplotlib.pyplot as plt
from skfuzzy import control as ctrl

# ============================================
# Step 3: Define the Universe (Ranges)
# ============================================
# Dirt Level: 0 to 100
# Load Size: 0 to 100
# Wash Time: 0 to 60 minutes

x_dirt = np.arange(0, 101, 1)
x_load = np.arange(0, 101, 1)
x_time = np.arange(0, 61, 1)

# ============================================
# Step 4: Define Fuzzy Variables
# ============================================
dirt = ctrl.Antecedent(x_dirt, 'dirt_level')
load = ctrl.Antecedent(x_load, 'load_size')
time = ctrl.Consequent(x_time, 'wash_time')

# ============================================
# Step 5: Define Membership Functions
# ============================================

# DIRT LEVEL - Triangular membership functions
dirt['Low'] = fuzz.trimf(x_dirt, [0, 0, 50])
dirt['Medium'] = fuzz.trimf(x_dirt, [25, 50, 75])
dirt['High'] = fuzz.trimf(x_dirt, [50, 100, 100])

# LOAD SIZE - Triangular membership functions
load['Small'] = fuzz.trimf(x_load, [0, 0, 50])
load['Medium'] = fuzz.trimf(x_load, [25, 50, 75])
load['Large'] = fuzz.trimf(x_load, [50, 100, 100])

# WASH TIME OUTPUT - Five membership functions
time['Very_Short'] = fuzz.trimf(x_time, [0, 0, 15])
time['Short'] = fuzz.trimf(x_time, [5, 15, 25])
time['Medium'] = fuzz.trimf(x_time, [20, 30, 40])
time['Long'] = fuzz.trimf(x_time, [35, 45, 55])
time['Very_Long'] = fuzz.trimf(x_time, [45, 60, 60])

# Set defuzzification method
time.defuzzify_method = 'centroid'

# ============================================
# Step 6: Define Fuzzy Rules (6 rules minimum)
# ============================================
rule1 = ctrl.Rule(dirt['Low'] & load['Small'], time['Very_Short'])
rule2 = ctrl.Rule(dirt['Low'] & load['Medium'], time['Short'])
rule3 = ctrl.Rule(dirt['Medium'] & load['Small'], time['Short'])
rule4 = ctrl.Rule(dirt['Medium'] & load['Medium'], time['Medium'])
rule5 = ctrl.Rule(dirt['High'] & load['Medium'], time['Long'])
rule6 = ctrl.Rule(dirt['High'] & load['Large'], time['Very_Long'])

# Additional rules for completeness
rule7 = ctrl.Rule(dirt['Low'] & load['Large'], time['Medium'])
rule8 = ctrl.Rule(dirt['Medium'] & load['Large'], time['Long'])
rule9 = ctrl.Rule(dirt['High'] & load['Small'], time['Medium'])

# Collect all rules
rules = [rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9]

# ============================================
# Step 7: Create Control System
# ============================================
washing_ctrl = ctrl.ControlSystem(rules)
washing_sim = ctrl.ControlSystemSimulation(washing_ctrl)

# ============================================
# Step 8: Display Membership Functions
# ============================================
print("=" * 50)
print("MEMBERSHIP FUNCTIONS VISUALIZATION")
print("=" * 50)

# Plot membership functions
dirt.view()
plt.title('Dirt Level Membership Functions')
plt.show()

load.view()
plt.title('Load Size Membership Functions')
plt.show()

time.view()
plt.title('Wash Time Membership Functions')
plt.show()

# ============================================
# Step 9: Test the System with Sample Inputs
# ============================================
print("\n" + "=" * 50)
print("TESTING THE FUZZY SYSTEM")
print("=" * 50)

# Test Case 1: Lightly soiled, small load
washing_sim.input['dirt_level'] = 20
washing_sim.input['load_size'] = 25
washing_sim.compute()
print(f"Test 1 - Dirt: 20, Load: 25 → Wash Time: {washing_sim.output['wash_time']:.1f} minutes")

# Test Case 2: Medium dirt, medium load
washing_sim.input['dirt_level'] = 50
washing_sim.input['load_size'] = 50
washing_sim.compute()
print(f"Test 2 - Dirt: 50, Load: 50 → Wash Time: {washing_sim.output['wash_time']:.1f} minutes")

# Test Case 3: Heavy dirt, large load
washing_sim.input['dirt_level'] = 85
washing_sim.input['load_size'] = 85
washing_sim.compute()
print(f"Test 3 - Dirt: 85, Load: 85 → Wash Time: {washing_sim.output['wash_time']:.1f} minutes")

# ============================================
# Step 10: Interactive User Input
# ============================================
print("\n" + "=" * 50)
print("INTERACTIVE MODE")
print("=" * 50)

try:
    user_dirt = float(input("Enter dirt level (0-100): "))
    user_load = float(input("Enter load size (0-100): "))

    washing_sim.input['dirt_level'] = user_dirt
    washing_sim.input['load_size'] = user_load
    washing_sim.compute()

    print(f"\n✅ Recommended Wash Time: {washing_sim.output['wash_time']:.1f} minutes")

except:
    print("Please enter valid numbers between 0-100")

# ============================================
# Step 11: 3D Surface Plot (Rule Viewer Output)
# ============================================
print("\n" + "=" * 50)
print("3D SURFACE PLOT - RULE VIEWER OUTPUT")
print("=" * 50)

# Create mesh grid for 3D visualization
dirt_vals = np.linspace(0, 100, 30)
load_vals = np.linspace(0, 100, 30)
D, L = np.meshgrid(dirt_vals, load_vals)
time_vals = np.zeros_like(D)

# Calculate wash time for each combination
for i in range(len(dirt_vals)):
    for j in range(len(load_vals)):
        washing_sim.input['dirt_level'] = D[i, j]
        washing_sim.input['load_size'] = L[i, j]
        washing_sim.compute()
        time_vals[i, j] = washing_sim.output['wash_time']

# Plot 3D surface
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(D, L, time_vals, cmap='viridis', edgecolor='none')

ax.set_xlabel('Dirt Level', fontsize=12)
ax.set_ylabel('Load Size', fontsize=12)
ax.set_zlabel('Wash Time (minutes)', fontsize=12)
ax.set_title('Fuzzy Logic Controller: Wash Time Surface Plot', fontsize=14)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Wash Time (min)')
plt.show()

print("\n" + "=" * 50)
print("IMPLEMENTATION COMPLETE!")
print("=" * 50)
print("✅ Membership functions displayed")
print("✅ 9 fuzzy rules implemented")
print("✅ Interactive testing available")
print("✅ 3D rule viewer output shown")